# HerBERT fine-tuning (Colab GPU)

Fine-tune **HerBERT** for 3-class Polish sentiment on **PolEmo 2.0**, reusing this project's tested code. **Runtime → Change runtime type → GPU** first.

This produces **two** files, and both are needed:

* `herbert.json` — the metrics, which fill the comparison table.
* `herbert_test.json` — the per-row predictions, in test-split order. Without these there is no paired McNemar test and no cascade: those panels compare *which reviews* the two models disagree on, and a score alone cannot answer that.

In [ ]:
!pip install -q "git+https://github.com/P0w3r223/pl-review-sense.git#egg=pl-review-sense[transformer]"

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())  # must be True

## Load data + fine-tune (full run)

In [ ]:
from pl_review_sense import data, herbert
ds = data.load_polemo()
result, y_pred = herbert.fine_tune(ds, herbert.full_args())
print(f'accuracy={result.accuracy:.4f}  macro_f1={result.macro_f1:.4f}')

## Save metrics (path-independent) and download

In [ ]:
import json, dataclasses
from pl_review_sense import config

payload = {
    'model': 'herbert', 'run': 'gpu', 'representative': True,
    'accuracy': result.accuracy, 'macro_f1': result.macro_f1,
    'per_class': [dataclasses.asdict(c) for c in result.per_class],
    'confusion': result.confusion, 'labels': [c.label for c in result.per_class],
}
with open('herbert.json', 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

# Same columnar shape and same row order as the baseline's predictions file: the pairing is
# positional, and nothing else records which review a row came from.
predictions = {
    'model': 'herbert',
    'labels': list(config.LABEL_NAMES),
    'true': [int(t) for t in ds.test.labels],
    'pred': [int(p) for p in y_pred],
}
with open('herbert_test.json', 'w', encoding='utf-8') as f:
    json.dump(predictions, f)
print('saved herbert.json and herbert_test.json')

try:
    from google.colab import files
    files.download('herbert.json')
    files.download('herbert_test.json')
except Exception:
    pass

**Then**, in the repo:

```bash
cp herbert.json      reports/metrics/herbert.json
cp herbert_test.json reports/predictions/herbert_test.json

python -m pl_review_sense.analysis   # re-runs the paired test and the cascade
python -m pl_review_sense.site       # rebuilds docs/index.html
```

Commit both the metrics and the rebuilt page: CI fails if the published page is not exactly what the committed metrics produce.